In [1]:
import glob
import os
import shutil
import zipfile
from pathlib import Path

WORK_DIR = Path("/kaggle/working/PA2")
WORK_DIR.parent.mkdir(parents=True, exist_ok=True)

def find_project_root():
    """Prefer the folder that has pyproject.toml AND scripts/train_sft.py."""
    hits = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
    good = [Path(h).parent for h in hits if (Path(h).parent / "scripts" / "train_sft.py").exists()]
    if good:
        return good[0]
    return Path(hits[0]).parent if hits else None

if (WORK_DIR / "pyproject.toml").exists():
    print("PA2 project already exists:", WORK_DIR)
else:
    root = find_project_root()
    if root is not None:
        print("Found extracted project:", root)
        shutil.copytree(root, WORK_DIR, dirs_exist_ok=True)
    else:
        zip_files = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        if not zip_files:
            raise FileNotFoundError("No pyproject.toml or .zip under /kaggle/input. Use Add Input to attach the PA2 dataset.")
        print("Extracting:", zip_files[0])
        with zipfile.ZipFile(zip_files[0]) as zf:
            zf.extractall("/kaggle/working")
        if not (WORK_DIR / "pyproject.toml").exists():
            found = [p for p in glob.glob("/kaggle/working/**/pyproject.toml", recursive=True)
                     if not p.startswith("/kaggle/working/PA2/")]
            if not found:
                raise FileNotFoundError("Extracted, but pyproject.toml not found. Inspect the ZIP structure.")
            shutil.copytree(str(Path(found[0]).parent), WORK_DIR, dirs_exist_ok=True)

os.chdir(WORK_DIR)
for folder in ("outputs/gen", "outputs/eval", "outputs/sft", "outputs/dpo_hh", "outputs/dpo_uf", "eval_sets"):
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project directory:", os.getcwd())
print("Has train_sft.py:", Path("scripts/train_sft.py").exists())
print("Top-level files:", sorted(os.listdir("."))[:30])


Found extracted project: /kaggle/input/datasets/areeshayaseen/cs5326-pa2/PA2
Project directory: /kaggle/working/PA2
Has train_sft.py: True
Top-level files: ['.gitignore', 'README.md', 'data', 'docs', 'eval_sets', 'outputs', 'pa2', 'pyproject.toml', 'requirements.txt', 'scripts', 'submission_adapter.py', 'tests']


**Install Dependencies**

In [2]:
%cd /kaggle/working/PA2

!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib
!python -m pip install -q -e . --no-deps

!nvidia-smi -L

import torch, transformers, datasets
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
assert torch.cuda.is_available(), "CUDA is unavailable. Set Accelerator to GPU T4 x2 in the notebook settings, then restart."


/kaggle/working
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for small-model-alignment-starter (p

**Public Adapter Smoke Tests**

In [3]:
%cd /kaggle/working/PA2
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py


/kaggle/working
.......                                                                  [100%]
7 passed in 1.95s


**SFT**

In [4]:
%cd /kaggle/working/PA2
!python scripts/train_sft.py --out_dir outputs/sft


/kaggle/working
README.md: 3.90kB [00:00, 20.5MB/s]
data/train_sft-00000-of-00003-a3ecf92756(…): 100%|█| 244M/244M [00:01<00:00, 122
data/train_sft-00001-of-00003-0a1804bcb6(…): 100%|█| 244M/244M [00:02<00:00, 118
data/train_sft-00002-of-00003-ee46ed25cf(…): 100%|█| 244M/244M [00:01<00:00, 173
data/test_sft-00000-of-00001-f7dfac4afe5(…): 100%|█| 81.2M/81.2M [00:01<00:00, 6
data/train_gen-00000-of-00003-a6c9fb894b(…): 100%|█| 244M/244M [00:01<00:00, 190
data/train_gen-00001-of-00003-d6a0402e41(…): 100%|█| 243M/243M [00:02<00:00, 104
data/train_gen-00002-of-00003-c0db75b92a(…): 100%|█| 243M/243M [00:01<00:00, 134
data/test_gen-00000-of-00001-3d4cd830914(…): 100%|█| 80.4M/80.4M [00:01<00:00, 4
Generating train_sft split: 100%|█| 207865/207865 [00:03<00:00, 52240.03 example
Generating test_sft split: 100%|█| 23110/23110 [00:00<00:00, 60016.42 examples/s
Generating train_gen split: 100%|█| 256032/256032 [00:03<00:00, 72988.46 example
Generating test_gen split: 100%|█| 28304/28304 [00:00<00:

**SFT Checkpoints verification**

In [5]:
%cd /kaggle/working/PA2
from pathlib import Path

best = Path("outputs/sft/best")
print("best dir exists:", best.exists())
if best.exists():
    for p in sorted(best.iterdir()):
        print(f"{p.name:35s} {p.stat().st_size/1e6:10.1f} MB")
print("config.json present:", (best / "config.json").exists())

!du -sh outputs/sft
!ls -la outputs/sft


/kaggle/working
best dir exists: True
added_tokens.json                          0.0 MB
chat_template.jinja                        0.0 MB
config.json                                0.0 MB
generation_config.json                     0.0 MB
merges.txt                                 1.7 MB
model.safetensors                       1260.4 MB
special_tokens_map.json                    0.0 MB
tokenizer.json                            11.4 MB
tokenizer_config.json                      0.0 MB
vocab.json                                 2.8 MB
config.json present: True
2.4G	outputs/sft
total 36
drwxr-xr-x 4 root root  4096 Oct 12 20:22 .
drwxr-xr-x 7 root root  4096 Oct 12 18:48 ..
drwxr-xr-x 2 root root  4096 Oct 12 20:22 best
drwxr-xr-x 2 root root  4096 Oct 12 20:22 final
-rw-r--r-- 1 root root 13685 Oct 12 20:22 metrics.jsonl
-rw-r--r-- 1 root root    57 Oct 12 18:49 run_config.json
